# INTERCON - code

I re-run this code

In [ ]:
!pip install -q transformers datasets accelerate sacrebleu sentencepiece

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/Colab Notebooks/Runasimi-MT"
print(base_path)

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/Runasimi-MT


In [ ]:
import torch

print(f"GPU available: {torch.cuda.is_available()}")
print(f"Number of GPUs available: {torch.cuda.device_count()}")
print(f"Name of GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU model: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU available: True
Number of GPUs available: 1
Name of GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU model: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 102.0 GB


In [ ]:
from datasets import load_dataset

dataset = load_dataset("somosnlp-hackathon-2022/spanish-to-quechua")
print(dataset)

README.md:   0%|          | 0.00/2.36k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/945 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 11.7MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.46MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating train split:   0%|          | 0/102747 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/12844 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12843 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['es', 'qu'],
        num_rows: 102747
    })
    validation: Dataset({
        features: ['es', 'qu'],
        num_rows: 12844
    })
    test: Dataset({
        features: ['es', 'qu'],
        num_rows: 12843
    })
})


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "facebook/nllb-200-distilled-600M",
    src_lang="spa_Latn",
    tgt_lang="quy_Latn"
)

def tokenize_function(examples):
    return tokenizer(
        examples["es"],
        text_target=examples["qu"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["es", "qu"])
print(tokenized_dataset)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

Map:   0%|          | 0/102747 [00:00<?, ? examples/s]

Map:   0%|          | 0/12844 [00:00<?, ? examples/s]

Map:   0%|          | 0/12843 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 102747
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 12844
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 12843
    })
})


In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, EarlyStoppingCallback
import os

output_dir = "/content/quechua_model"  # save colab session
# output_dir = "/content/drive/MyDrive/Colab Notebooks/Runasimi-MT/quechua_model_v2" # save in drive

os.makedirs(output_dir, exist_ok=True)

model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    weight_decay=0.01,
    fp16=True,
    predict_with_generate=True,
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.tokenizer = tokenizer

trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.46GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

... (9 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=64220, training_loss=1.1067072959081363, metrics={'train_runtime': 12133.8887, 'train_samples_per_second': 84.678, 'train_steps_per_second': 5.293, 'total_flos': 2.7832937069150208e+17, 'train_loss': 1.1067072959081363, 'epoch': 10.0})

In [ ]:
import torch
import sacrebleu
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset
import os
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

output_dir = "/content/quechua_model" # save colab session
# output_dir = "/content/drive/MyDrive/Colab Notebooks/Runasimi-MT/quechua_model" # save in drive

# Find best checkpoint, using lowest eval_loss
checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
best_cp = None
best_loss = float("inf")
for cp in checkpoints:
    state_path = os.path.join(output_dir, cp, "trainer_state.json")
    if os.path.exists(state_path):
        with open(state_path, "r") as f:
            state = json.load(f)
        best = state.get("best_metric", None)
        if best is not None and best < best_loss:
            best_loss = best
            best_cp = cp
if best_cp is None:
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    best_cp = checkpoints[-1]
print(f"Best checkpoint: {best_cp} (eval_loss={best_loss:.4f})")

# Load model and tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(os.path.join(output_dir, best_cp)).to(device)
tokenizer = AutoTokenizer.from_pretrained(
    "facebook/nllb-200-distilled-600M",
    src_lang="spa_Latn",
    tgt_lang="quy_Latn"
)

# Load dataset
dataset = load_dataset("somosnlp-hackathon-2022/spanish-to-quechua")

# Batched translation
def translate_batch(texts, batch_size=32):
    model.eval()
    translations = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True, max_length=128, padding=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=128, num_beams=4)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
    return translations

test_es = dataset["test"]["es"]
test_qu = dataset["test"]["qu"]

preds = translate_batch(test_es)
refs = [[ref] for ref in test_qu]

bleu = sacrebleu.corpus_bleu(preds, refs)
chrf = sacrebleu.corpus_chrf(preds, refs, word_order=2)

print(f"Full test set (12,843 examples)")
print(f"BLEU: {bleu.score:.2f}")
print(f"chrF++: {chrf.score:.2f}")

# Optionally save predictions for analysis
with open(os.path.join(output_dir, "full_test_predictions.txt"), "w") as f:
    for pred, ref in zip(preds, test_qu):
        f.write(f"PRED: {pred}\nREF: {ref}\n\n")
print("Predictions saved.")

Best checkpoint: checkpoint-64220 (eval_loss=0.2534)


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Translating: 100%|██████████| 402/402 [04:08<00:00,  1.62it/s]


Full test set (12,843 examples)
BLEU: 18.58
chrF++: 43.58
Predictions saved.


In [ ]:
# Evaluate with epoch 9 (checkpoint-57798)
model = AutoModelForSeq2SeqLM.from_pretrained(os.path.join(output_dir, "checkpoint-57798")).to(device)

preds = translate_batch(test_es)
refs = [[ref] for ref in test_qu]

bleu = sacrebleu.corpus_bleu(preds, refs)
chrf = sacrebleu.corpus_chrf(preds, refs, word_order=2)

print(f"Test set (12,843 examples) — checkpoint-57798")
print(f"BLEU: {bleu.score:.2f}")
print(f"chrF++: {chrf.score:.2f}")

with open(os.path.join(output_dir, "test_predictions_ckpt57798.txt"), "w") as f:
    for pred, ref in zip(preds, test_qu):
        f.write(f"PRED: {pred}\nREF: {ref}\n\n")
print("Predictions saved.")

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Translating: 100%|██████████| 402/402 [04:08<00:00,  1.62it/s]


Test set (12,843 examples) — checkpoint-57798
BLEU: 22.89
chrF++: 40.87
Predictions saved.


## Reproducibility notes

Re-running the pipeline gives slightly different results from the paper due to non-determinism (fp16, CUDA, library versions). Two checkpoints from the same run were evaluated:

| Source | Checkpoint | BLEU | chrF++ |
| :--- | :--- | :--- | :--- |
| Paper | step 57,798 (epoch 9) | 17.67 | 42.68 |
| Reproduction | checkpoint-57798 (epoch 9) | 22.89 | 40.87 |
| Reproduction | checkpoint-64220 (epoch 10) | 18.58 | 43.58 |

checkpoint-64220 is closest to the paper on both metrics (Δ ≈ 0.9), consistent with normal non-determinism. checkpoint-57798 shows a different BLEU/chrF++ trade-off, reflecting the known decoupling between validation loss and generation metrics.

To continue this research, we will implement full reproducibility so these variations no longer appear: fixed random seed (`seed`, `data_seed`, `full_determinism`), pinned library versions, corrected checkpoint selection, and model selection by validation chrF++ instead of validation loss. Until then, the reported numbers should be read as one draw from a distribution rather than a fixed result.